# MOA Microlensing Catalog Search

Check whether a sky coordinate has been observed by the **MOA** (Microlensing Observations in
Astrophysics) survey image catalog, and if so retrieve its reference-image cutout and
extract an aperture photometry light curve - all via the `api-vera-rubin-v1` process API.

**Outputs**
- A 15 arcsec cutout of the reference image around the closest detected source
- Photometry for the selected source (mid-exposure JD, aperture flux, and flux error)
- The cutout saved as FITS and the light curve saved as CSV, under `OUTPUT_DIR`

**Expected runtime:** the coverage check takes up to ~3 minutes; photometry runs as a
job on the server over the full archive, roughly 30 s per 1000 observations.

**References**
- MOA Catalog API docs: https://moacatalog.auckland.ac.nz/docs
- Sumi et al. (2002): https://arxiv.org/abs/astro-ph/0207604
- Sako et al. (2008): https://arxiv.org/abs/0804.0653
- Baldwin et al. (2026): https://iopscience.iop.org/article/10.3847/2515-5172/ae353e 

**Contents**

0. Setup & parameters
1. Check coordinates 
2. Aperture photometry
3. Save outputs
4. Reload a saved CSV
5. Results & next steps

## 0. Setup

Imports, and the `lib/` path bootstrap so `moa_catalog_client` resolves regardless of where Jupyter set the kernel's working directory.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


def _find_lib_dir(start: Path, marker: str = "moa_catalog_client.py") -> Path:
    for candidate in (start, *start.parents):
        lib_dir = candidate / "lib"
        if (lib_dir / marker).exists():
            return lib_dir
    raise RuntimeError(f"Could not locate lib/ (looked for {marker}) above {start}")


LIB_DIR = _find_lib_dir(Path.cwd())
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

In [ ]:
import csv
import getpass
import os

import ipywidgets as widgets
import matplotlib.pyplot as plt
from astropy.nddata import CCDData
from astropy.wcs.utils import skycoord_to_pixel
from IPython.display import display

from moa_catalog_client import (
    API_BASE_URL,
    AUTH_APP_URL,
    MAX_BKG_OUT,
    check_in_moa,
    define_annulus,
    display_cutout,
    get_photometry_job,
    load_photometry_csv,
    parse_coordinates,
    plot_lightcurve,
    retrieve_photometry,
    set_access_token,
    validate_photometry,
)

### Parameters

Everything you'd normally change lives in the cell below.

- RA as `hh mm ss.ss` or decimal degrees; Dec as `(+/-)dd mm ss.ss` or decimal degrees
- `COORD_FRAME`: `icrs`, `fk4` or `fk5`
- Aperture radii are in pixels: `1 <= STAR_RADIUS <= 5` and `STAR_RADIUS < BKG_IN <= BKG_OUT <= 7` (checked in section 2a)

In [ ]:
# Target
TARGET_RA = "00 47 22.11"
TARGET_DEC = "-73 06 54.02"
COORD_FRAME = "fk5"

# Aperture (pixels)
STAR_RADIUS = 1.0
BKG_IN = 1.5
BKG_OUT = 2.0

# Photometry always runs over 100% of the archive - the only value the API accepts.
# Set to a job id printed by an earlier run to collect that job's result instead
# of submitting a new job (see section 2b).
RESUME_JOB_ID = None
OUTPUT_DIR = "outputs"

# Optional demo of an unobserved coordinate (adds ~3 minutes)
RUN_NOT_OBSERVED_DEMO = False
UNOBSERVED_RA = "10 17 9.9"
UNOBSERVED_DEC = "-15 15 9.9"

#### Example targets

| RA | Dec | Frame | Field | Notes |
|---|---|---|---|---|
| `268.1204134350` | `-28.8896223900` | fk5 | gb5-2 (bulge) | Dense field: ~1217 obs (2007), ~34740 in the full archive, so photometry takes 15+ minutes |
| `274.8818981657` | `-24.7913523900` | fk5 | bulge | |
| `17 52 25.704` | `-29 56 19.131` | fk5 | bulge | Sexagesimal input |
| `00 47 22.11` | `-73 06 54.02` | fk5 | SMC | Default. Smaller field, shorter photometry job |
| `75.6167093108` | `-69.4606949100` | fk5 | lmc7-4 | |
| `8.5274187135` | `-75.32129405` | fk5 | SMC | |
| `5.9026440207` | `-74.6221151200` | fk5 | SMC | ⚠️ Too faint to be detected in MOA images |
| `267.2987926888` | `-34.5928878300` | fk5 | bulge | ⚠️ Photometry fails at ~90%: *Empty or corrupt FITS file* |
| `11.3647760272` | `-72.5625561300` | fk5 | smc1-2 | ⚠️ *Empty or corrupt FITS file* |

### Authentication

Calls to the MOA Catalog API ([docs](https://moacatalog.auckland.ac.nz/docs)) require a Cognito **access token**.

1. Open the auth app (https://moacatalog.auckland.ac.nz) and sign in. The cell below prints its URL.
2. Copy the **access token** from the dashboard (not the ID token).
3. Run the cell below and paste it at the prompt - or, for headless runs, set the
   `VERA_RUBIN_ACCESS_TOKEN` environment variable before starting Jupyter.

The token is held in memory for this kernel session only and is never printed or
written to disk. Access tokens are short-lived: if a later call fails with a 401, get
a fresh token and re-run this cell.

In [ ]:
print(f"MOA Catalog API: {API_BASE_URL}")
if os.environ.get("VERA_RUBIN_ACCESS_TOKEN"):
    set_access_token(os.environ["VERA_RUBIN_ACCESS_TOKEN"])
    print("Access token loaded from VERA_RUBIN_ACCESS_TOKEN.")
else:
    set_access_token(getpass.getpass(f"Paste your Cognito access token from {AUTH_APP_URL} : "))

## 1. Check Coordinates 

### 1a. (Optional) A coordinate MOA has not observed

Shows what an unobserved coordinate looks like. Skipped unless
`RUN_NOT_OBSERVED_DEMO = True`, because it costs ~3 minutes.

In [ ]:
if RUN_NOT_OBSERVED_DEMO:
    unobserved_coords = parse_coordinates(UNOBSERVED_RA, UNOBSERVED_DEC, "fk5")
    check_in_moa(unobserved_coords)
else:
    print("Skipped (RUN_NOT_OBSERVED_DEMO = False).")

### 1b. Check the target

`check_in_moa` returns:

1. a [`Cutout2D`](https://docs.astropy.org/en/stable/api/astropy.nddata.Cutout2D.html)
   of the reference image, carrying its WCS;
2. a [`SkyCoord`](https://docs.astropy.org/en/stable/api/astropy.coordinates.SkyCoord.html)
   for the closest detected source;
3. the reference image's FITS header.

If the target was observed, a 15 arcsec cutout is displayed with a circle on the
closest source. Otherwise the run stops here with an explanation.

In [ ]:
target_coords = parse_coordinates(TARGET_RA, TARGET_DEC, COORD_FRAME)
image_cutout, source_coords, header_data = check_in_moa(target_coords)

if image_cutout is None or source_coords is None:
    raise RuntimeError(
        "No usable MOA data for the target: it was not observed, no source cleared the "
        "5-sigma detection threshold, or the request failed (e.g. a 401 - see above)."
    )

print(f"Closest source: {source_coords.to_string('decimal', precision=6)}")
source_x, source_y = skycoord_to_pixel(source_coords, image_cutout.wcs)
display_cutout(image_cutout, (source_x, source_y))
plt.show()

### 1c. Reference header

The FITS header included with the cutout.

In [ ]:
print(repr(header_data))

## 2. Aperture Photometry

Aperture photometry is a technique used to measure the integrated flux of a source. 

* **Source Aperture:** A circular aperture (radius in pixels) centered on the source's centroid. Summing the pixel values within this region captures the total flux of the object.
* **Background Contribution:** All images contain additive background flux so that every pixel within the source aperture records both the target flux and this background.
* **Key Calculation:** Aperture photometry calculates the net source flux by subtracting the estimated background contribution multiplied by the area of the source aperture.


### 2a. Define annulus

To perform accurate background subtraction, we must identify the concentric, donut-shaped region surrounding the source aperture. Properly defining the inner and outer radii of this annulus is critical for photometric precision.

Sky backgrounds are rarely uniform across an astronomical frame due to a variety of factors. An annulus placed immediately adjacent to the aperture measures the local background baseline specific to the desired coordinate.

**<u>Inner Radius Selection (BKG_IN)</u>**
* **Function:** Excludes the extended region of the star's PSF.
* **Risk of Incorrect Sizing:** 
  * If the inner radius is too small, the annulus intersects the outer edges of the source flux. Treating this light as background results in over-subtraction and an artificially lowered flux measurement.
  * If the inner radius is unnecessarily large, it moves away from the immediate vicinity of the target, degrading the representation of the local background.

**<u>Outer Radius Selection (BKG_OUT)</u>**
* **Function:** Determines the total area sampled for background statistics.
* **Risk of Incorrect Sizing:** 
  * The outer radius must encompass enough pixels to yield a statistically stable background estimate.
  * However, extending it too far risks contamination from neighboring astronomical sources and artefacts.

In [ ]:
assert 1.0 <= STAR_RADIUS <= 5.0, "STAR_RADIUS must be between 1 and 5 pixels"
assert STAR_RADIUS < BKG_IN <= BKG_OUT <= MAX_BKG_OUT, (
    f"Need STAR_RADIUS < BKG_IN <= BKG_OUT <= {MAX_BKG_OUT}"
)

define_annulus(image_cutout, (source_x, source_y), STAR_RADIUS, BKG_IN, BKG_OUT)
plt.show()

### 2b. Retrieve photometry

`retrieve_photometry` submits a photometry job to the process API, which always runs
over the full MOA archive. It then checks the job's status every minute, printing each
change, and downloads the result once the job finishes. Expect roughly 30 s per 1000
observations, so a dense bulge field can take 15 minutes or more.

You can also follow the job in the MOA Catalog app at `https://moacatalog.auckland.ac.nz/jobs/<job_id>`; the link is printed when the job is submitted.

The result is an array with one row per observation: column 0 is the mid-exposure
Julian date, column 1 the aperture flux, and column 2 its associated uncertainty (the API sends
these as plain `jd,flux,uncertainty` CSV rows). `nan` values are returned where flux retrival failed. 

> **Note:** the notebook stops waiting after 1 hour, or when you interrupt the
> kernel, but the job keeps running on the server. To collect its result without
> submitting a new job (submissions are limited to 3 per minute and 15 per hour), set
> `RESUME_JOB_ID` in the Parameters cell to the printed job id and re-run this section.

In [ ]:
if RESUME_JOB_ID:
    moa_photometry = get_photometry_job(RESUME_JOB_ID)
else:
    moa_photometry = retrieve_photometry(
        source_coords,
        star_radius=STAR_RADIUS,
        bkg_in=BKG_IN,
        bkg_out=BKG_OUT,
    )
validate_photometry(moa_photometry)
print(f"{len(moa_photometry)} observations passed validation.")

In [ ]:
plot_lightcurve(moa_photometry)
plt.show()

## 3. Save outputs

The cutout (with the reference header) is saved as FITS, and the light curve as a
CSV with a header row (`jd_mid,aperture_sum,uncertainty`). Both go to `OUTPUT_DIR`,
which is gitignored.

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
file_stem = f"{source_coords.ra.degree:.4f}_{source_coords.dec.degree:.4f}"

fits_path = output_dir / f"my_source_{file_stem}.fits"
CCDData(data=image_cutout.data, unit="adu", header=header_data).write(fits_path, overwrite=True)

csv_path = output_dir / f"fluxes_{file_stem}_moa.csv"
with open(csv_path, "w", newline="") as outfile:
    csv_writer = csv.writer(outfile)
    csv_writer.writerow(["jd_mid", "aperture_sum", "uncertainty"])
    csv_writer.writerows(moa_photometry)

print(f"Saved {fits_path} and {csv_path}")

## 4. Reload a saved CSV (optional)

Upload a previously saved fluxes CSV (e.g. `outputs/fluxes_268.1210_-28.8895_moa.csv`)
to reproduce the light curve without re-running the search and photometry. Older
CSVs without a header row also work.

In [ ]:
uploader = widgets.FileUpload(accept=".csv", multiple=False)
display(uploader)

In [ ]:
if not uploader.value:
    print("No file uploaded - use the widget above to select a CSV, then re-run this cell.")
else:
    uploaded = uploader.value
    uploaded_file = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
    uploaded_photometry = load_photometry_csv(bytes(uploaded_file["content"]))
    validate_photometry(uploaded_photometry)
    plot_lightcurve(uploaded_photometry)
    plt.show()

## 5. Results and next steps

- **Long jobs:** if section 2b stopped waiting before the job finished, set
  `RESUME_JOB_ID` to the printed job id and re-run section 2b to collect the result.
- **Saved files:** the cutout and light curve are in `OUTPUT_DIR`. Reload the CSV
  with section 4 at any time.